In [1]:
from IPython.display import clear_output
from Fetch_Live_Data import *
from Buy_Presure_Scaner import *
from Trade_Execution import *
from Chandelier_ZLSMA import *
from Chandelier_ZLSMA_Filter import *
from EMAs_9_15_Filter import *
from Fetch_Coin_List import *

In [59]:
#coins = get_symbol(max_price=30, min_age_days=365, min_data_days=365)
#print("Total Coin Found: ", len(coins))

In [60]:
df = pd.read_csv("Coin_List.csv", header=None)
coins = df[1:].iloc[:, 0].tolist()
print("Total Coins:", len(coins))

def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 500)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[480:]

Total Coins: 291


Fetching BTC/USDT data from Binance...
BTC Chandler Exit Buy Signal is BTC Trend is Bearish, exiting the scan...


In [6]:
import logging
import pandas as pd
from typing import List, Optional
from IPython.display import clear_output  # Assuming Jupyter environment
import time
import signal
import sys

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def run_scan(
    coins: List[str],
    timeframe: str = '15m',
    max_buy_candles: int = 3,
    zlsma_length: int = 200,
    btc_index: int = 199,
    btc_sum_start: int = 198
) -> List[str]:
    """
    Run a cryptocurrency trading scan to identify bullish coins based on BTC trend and chandelier signals.

    Args:
        coins (List[str]): List of coin symbols to scan.
        timeframe (str): Timeframe for analysis (default: '15m').
        max_buy_candles (int): Maximum number of buy candles allowed (default: 3).
        zlsma_length (int): Length for ZLSMA calculation (default: 200).
        btc_index (int): Index for checking BTC buy signal (default: 199).
        btc_sum_start (int): Start index for summing BTC buy signals (default: 198).

    Returns:
        List[str]: List of symbols that pass the bullish scan criteria.
    """
    clear_output(wait=True)
    out_coins: List[str] = []

    try:
        btc_data = only_chandelier('BTC')
        if not isinstance(btc_data, pd.DataFrame) or 'buy_signal' not in btc_data.columns:
            logger.error("Invalid BTC data format")
            return []

        if len(btc_data) <= btc_index:
            logger.error(f"BTC data has insufficient rows: {len(btc_data)} < {btc_index + 1}")
            return []

        if btc_data['buy_signal'].iloc[btc_index] != 0:
            btc_out = btc_data['buy_signal'].iloc[btc_sum_start:].sum()
            if btc_out >= 2:
                logger.info("🐂🟢 BTC Trend is Bullish, Proceeding with the scan...")
                
                chandelier_zlsma_out = chandelier_zlsma_filter_custom(
                    coin_list=coins,
                    timeframe=timeframe,
                    max_buy_candles=max_buy_candles,
                    zlsma_length=zlsma_length
                )

                if not isinstance(chandelier_zlsma_out, pd.DataFrame) or chandelier_zlsma_out.empty:
                    logger.warning("🔴 No coins found in 1st phase, 🔍Scan Continue...")
                    return []

                buy_pressure = get_binance_buy_pressure(
                    chandelier_zlsma_out['symbol'].tolist(),
                    top_n=5
                )
                logger.info("Second Phase...")
                chandelier_zlsma_out = chandelier_zlsma_filter_custom(
                    coin_list=buy_pressure['symbol'].tolist(),
                    timeframe=timeframe,
                    max_buy_candles=max_buy_candles,
                    zlsma_length=zlsma_length
                )

                if not isinstance(chandelier_zlsma_out, pd.DataFrame) or chandelier_zlsma_out.empty:
                    logger.warning("🔴 No coins found in 2nd phase, 🔍Scan Continue...")
                    return []

                for symbol in chandelier_zlsma_out['symbol'].tolist():
                    logger.info(f"Processing symbol: {symbol}")
                    try:
                        out = fetch_and_process_data(symbol)
                        if not isinstance(out, pd.DataFrame) or 'buy_signal' not in out.columns:
                            logger.error(f"Invalid data format for symbol: {symbol}")
                            continue

                        zero_signals = out[out['buy_signal'] == 0]
                        if zero_signals.empty:
                            logger.warning(f"No zero buy signals for {symbol}, skipping...")
                            continue

                        last_zero_index = zero_signals.index[-1]
                        count_after_last_zero = out.loc[last_zero_index + 1:, 'buy_signal'].sum()
                        buy_candles = chandelier_zlsma_out.loc[
                            chandelier_zlsma_out['symbol'] == symbol,
                            'buy_candles_count'
                        ].iloc[0]

                        if buy_candles == count_after_last_zero:
                            logger.info("Third Phase...")
                            symbol_data = only_chandelier(symbol)
                            if not isinstance(symbol_data, pd.DataFrame) or 'buy_signal' not in symbol_data.columns:
                                logger.error(f"Invalid chandelier data for symbol: {symbol}")
                                continue
                            if len(symbol_data) > btc_index and symbol_data['buy_signal'].iloc[btc_index] != 0:
                                out_coins.append(symbol)

                    except Exception as e:
                        logger.error(f"Error processing symbol {symbol}: {str(e)}")
                        continue

                return out_coins

            else:
                logger.warning("🔴🐻 BTC Trend Not Enough, 🔍Scan Continue...")
                return []

        else:
            logger.warning("🔴🐻 BTC Trend is Bearish, 🔍Scan Continue...")
            return []

    except Exception as e:
        logger.error(f"Error in run_scan: {str(e)}")
        return []

def run_scan_with_retry(
    coins: List[str],
    retry_interval: float = 30.0,
    timeframe: str = '15m',
    max_buy_candles: int = 3,
    zlsma_length: int = 200,
    btc_index: int = 199,
    btc_sum_start: int = 198
) -> List[str]:
    """
    Run the scan function with automatic retries every 30 seconds if BTC trend is bearish.

    Args:
        coins (List[str]): List of coin symbols to scan.
        retry_interval (float): Seconds to wait between retries (default: 30.0).
        timeframe (str): Timeframe for analysis (default: '15m').
        max_buy_candles (int): Maximum number of buy candles allowed (default: 3).
        zlsma_length (int): Length for ZLSMA calculation (default: 200).
        btc_index (int): Index for checking BTC buy signal (default: 199).
        btc_sum_start (int): Start index for summing BTC buy signals (default: 198).

    Returns:
        List[str]: List of symbols that pass the bullish scan criteria.
    """
    def signal_handler(sig, frame):
        logger.info("Scan interrupted by user.")
        sys.exit(0)

    signal.signal(signal.SIGINT, signal_handler)

    while True:
        logger.info(f"Starting scan at {time.strftime('%Y-%m-%d %H:%M:%S %Z')}")
        result = run_scan(
            coins=coins,
            timeframe=timeframe,
            max_buy_candles=max_buy_candles,
            zlsma_length=zlsma_length,
            btc_index=btc_index,
            btc_sum_start=btc_sum_start
        )

        if result:
            logger.info(f"Scan completed successfully. Bullish coins: {result}")
            return result

        logger.info(f"Retrying scan in {retry_interval} seconds...")
        time.sleep(retry_interval)

In [7]:
# Example usage
if __name__ == "__main__":
    try:
        bullish_coins = run_scan_with_retry(
            coins=coins,
            retry_interval=30.0
        )
        print(f"Final bullish coins: {bullish_coins}")
    except KeyboardInterrupt:
        print("Program terminated by user.")

Fetching BTC/USDT data from Binance...


2025-06-18 17:48:56,329 - WARNING - 🔴🐻 BTC Trend is Bearish, 🔍Scan Continue...
2025-06-18 17:48:56,329 - INFO - Retrying scan in 30.0 seconds...
2025-06-18 17:48:59,010 - INFO - Scan interrupted by user.


SystemExit: 0

In [ ]:
bullish_coins

['AEVOUSDT', 'MDTUSDT', 'SUNUSDT', 'ALPINEUSDT', 'AUDIOUSDT']

In [10]:
import logging
import pandas as pd
from typing import List, Optional, Union
from IPython.display import clear_output
import time
import signal
import sys

df = pd.read_csv("Coin_List.csv", header=None)
coins = df[1:].iloc[:, 0].tolist()
print("Total Coins:", len(coins))

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def run_scan(
    coins: List[str],
    timeframe: str = '15m',
    max_buy_candles: int = 3,
    btc_index: int = 199,
    btc_sum_start: int = 198
) -> Union[pd.DataFrame, List[str]]:
    """
    Run a cryptocurrency trading scan to identify bullish coins based on BTC trend.

    Args:
        coins (List[str]): List of coin symbols to scan.
        timeframe (str): Timeframe for analysis (default: '15m').
        max_buy_candles (int): Maximum number of buy candles allowed (default: 3).
        btc_index (int): Index for checking BTC buy signal (default: 199).
        btc_sum_start (int): Start index for summing BTC buy signals (default: 198).

    Returns:
        Union[pd.DataFrame, List[str]]: DataFrame with bullish coins or empty list if no coins found.
    """
    clear_output(wait=True)
    out_coins: List[str] = []

    try:
        btc_data = only_chandelier('BTC')
        if not isinstance(btc_data, pd.DataFrame) or 'buy_signal' not in btc_data.columns:
            logger.error("Invalid BTC data format")
            return []

        if len(btc_data) <= btc_index:
            logger.error(f"BTC data has insufficient rows: {len(btc_data)} < {btc_index + 1}")
            return []

        if btc_data['buy_signal'].iloc[btc_index] != 0:
            btc_out = btc_data['buy_signal'].iloc[btc_sum_start:].sum()
            if btc_out >= 2:
                logger.info("🐂🟢 BTC Trend is Bullish, Proceeding with the scan...")
                
                logger.info("First Phase...")
                EMAs_out = EMAs_9_15_Filter(
                    symbols=coins,
                    timeframe='15m',
                    max_cross_candles=5,
                    max_price=50,
                    show_all=False
                )

                if not isinstance(EMAs_out, pd.DataFrame) or EMAs_out.empty:
                    logger.warning("🔴 No coins found in 1st phase, 🔍Scan Continue...")
                    return []

                buy_pressure = get_binance_buy_pressure(
                    EMAs_out['symbol'].tolist(),
                    top_n=5
                )
                logger.info("Second Phase...")
                EMAs_out = EMAs_9_15_Filter(
                    symbols=buy_pressure['symbol'].tolist(),
                    timeframe='15m',
                    max_cross_candles=5,
                    max_price=50,
                    show_all=False
                )

                if not isinstance(EMAs_out, pd.DataFrame) or EMAs_out.empty:
                    logger.warning("🔴 No coins found in 2nd phase, 🔍Scan Continue...")
                    return []

                return EMAs_out['symbol'].tolist()

            else:
                logger.warning("🔴🐻 BTC Trend Not Enough, 🔍Scan Continue...")
                return []

        else:
            logger.warning("🔴🐻 BTC Trend is Bearish, 🔍Scan Continue...")
            return []

    except Exception as e:
        logger.error(f"Error in run_scan: {str(e)}")
        return []

def run_scan_with_retry(
    coins: List[str],
    retry_interval: float = 30.0,
    timeframe: str = '15m',
    max_buy_candles: int = 3,
    btc_index: int = 199,
    btc_sum_start: int = 198
) -> Union[pd.DataFrame, List[str]]:
    """
    Run the scan function with automatic retries every 30 seconds if BTC trend is bearish.

    Args:
        coins (List[str]): List of coin symbols to scan.
        retry_interval (float): Seconds to wait between retries (default: 30.0).
        timeframe (str): Timeframe for analysis (default: '15m').
        max_buy_candles (int): Maximum number of buy candles allowed (default: 3).
        btc_index (int): Index for checking BTC buy signal (default: 199).
        btc_sum_start (int): Start index for summing BTC buy signals (default: 198).

    Returns:
        Union[pd.DataFrame, List[str]]: DataFrame with bullish coins or empty list if no coins found.
    """
    def signal_handler(sig, frame):
        logger.info("Scan interrupted by user.")
        sys.exit(0)

    signal.signal(signal.SIGINT, signal_handler)

    while True:
        logger.info(f"Starting scan at {time.strftime('%Y-%m-%d %H:%M:%S %Z')}")
        result = run_scan(
            coins=coins,
            timeframe=timeframe,
            max_buy_candles=max_buy_candles,
            btc_index=btc_index,
            btc_sum_start=btc_sum_start
        )

        # Fix: Check if result is not empty instead of using truthiness
        if isinstance(result, pd.DataFrame) and not result.empty:
            logger.info(f"Scan completed successfully. Found {len(result)} bullish coins")
            return result
        elif isinstance(result, list) and result:
            logger.info(f"Scan completed successfully. Bullish coins: {result}")
            return result

        logger.info(f"Retrying scan in {retry_interval} seconds...")
        time.sleep(retry_interval)

Total Coins: 291


In [ ]:
df = run_scan_with_retry(coins)

Fetching BTC/USDT data from Binance...


In [ ]:
out_coins = []
coin_analysis = []
btc_data = only_chandelier('BTC')

if btc_data['buy_signal'].iloc[199] != 0:
    print("buy_signal is not zero, proceeding with the scan...")
    # Sum from the 199th element onward (position 198 and beyond)
    btc_out = btc_data['buy_signal'].iloc[198:].sum()
    if btc_out >= 2:
        print("BTC Trend is Bullish, Proceeding with the scan...")
        chandelier_zlsma_out = chandelier_zlsma_filter_custom(
            coin_list=coins,
            timeframe='15m',
            max_buy_candles=3,
            zlsma_length=200
        )
        
        if len(chandelier_zlsma_out) == 0:
            print("No coins found in the first phase, exiting the scan...")
        else:
            buy_pressure = get_binance_buy_pressure(chandelier_zlsma_out['symbol'].tolist(), top_n=5)
            
            print("Second Phase...")
            
            chandelier_zlsma_out = chandelier_zlsma_filter_custom(
                coin_list=buy_pressure['symbol'].tolist(),
                timeframe='15m',
                max_buy_candles=3,
                zlsma_length=200
            )
            
            for symbol in chandelier_zlsma_out['symbol'].tolist():
                print(f"Processing symbol: {symbol}")
                out = fetch_and_process_data(symbol)
                coin_analysis.append(out)
                
                last_zero_index = out[out['buy_signal'] == 0].index[-1]
                count_after_last_zero = out.loc[last_zero_index + 1:, 'buy_signal'].sum()
                if chandelier_zlsma_out[chandelier_zlsma_out['symbol'] == symbol]['buy_candles_count'].tolist()[0] == count_after_last_zero:
                    print("Third Phase...")
                    btc_data = only_chandelier(symbol)
                    if btc_data['buy_signal'].iloc[199] != 0:
                    
                        out_coins.append(symbol)
    else:
        print("BTC Trend is Bearish, Proceeding with the scan...")
else:
    print("BTC Chandler Exit Buy Signal is BTC Trend is Bearish, exiting the scan...")

In [ ]:
while True:
    clear_output(wait=True)  # Clear previous output before printing new scan result
    btc_data = calculate_crypto_chandelier('BTC')
    if btc_data['buy_signal'][199] != 0:
        if btc_out = btc_data[198:]['buy_signal'].sum() >= 2:
        

        if btc_out == False:
            print("BTC In Downtrend, Exiting the scan...")
            time.sleep(10)
            continue
        else:

            # Proceed with scan if BTC is bullish
            print("BTC Trend is Bullish, Proceeding with the scan...")
            chandelier_zlsma_out = chandelier_zlsma_filter_custom(
                coin_list=coins,
                timeframe='15m',
                max_buy_candles=3,
                zlsma_length=200
            )
            
            if len(chandelier_zlsma_out) == 0:
                print("No coins found in the first phase, exiting the scan...")
                time.sleep(10)
                continue  # Skip the rest and restart loop
            
            buy_pressure = get_binance_buy_pressure(chandelier_zlsma_out['symbol'].tolist(), top_n=5)
            
            print("Second Phase...")
            
            chandelier_zlsma_out = chandelier_zlsma_filter_custom(
                coin_list=buy_pressure['symbol'].tolist(),
                timeframe='15m',
                max_buy_candles=3,
                zlsma_length=200
            )
            
            out_coins = []
            coin_analysis = []
            for symbol in chandelier_zlsma_out['symbol'].tolist():
                print(f"Processing symbol: {symbol}")
                out = fetch_and_process_data(symbol)
                coin_analysis.append(out)
                
                last_zero_index = out[out['buy_signal'] == 0].index[-1]
                count_after_last_zero = out.loc[last_zero_index + 1:, 'buy_signal'].sum()
                if chandelier_zlsma_out[chandelier_zlsma_out['symbol'] == symbol]['buy_candles_count'].tolist()[0] == count_after_last_zero:
                    print("Third Phase...")
                    symbol_filtered = chandelier_zlsma_filter_custom(
                        coin_list=symbol,
                        timeframe='15m',
                        max_buy_candles=3,
                        zlsma_length=200
                    )
                    
                    out_coins.append(symbol_filtered['symbol'].tolist())
    
    time.sleep(10)  # Wait before next scan

Fetching BTC data from Binance...
Fetched 200 candles
Calculating Chandelier Exit...
BTC Trend is Bullish, Proceeding with the scan...
Step 1: Scanning for Chandelier Exit buy signals...
Step 2: Filtering by ZLSMA(200)...
Second Phase...
Step 1: Scanning for Chandelier Exit buy signals...
Step 2: Filtering by ZLSMA(200)...
Processing symbol: LTOUSDT
Third Phase...
Step 1: Scanning for Chandelier Exit buy signals...
Step 2: Filtering by ZLSMA(200)...


KeyError: 'symbol'

In [76]:
symbol_filtered

""


In [ ]:
# Define your function to fetch, calculate, and merge the data
def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 500)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[480:]

def run_task():
    # Get the latest symbols every 2 hours
    symbols = pick_best_coin()
    return symbols

def fetch_and_display_data_(symbols):
    # Run the fetch_and_process_data every 10 seconds to display results
    result = fetch_and_process_data(symbols)
    clear_output(wait=True)  # Clear previous output in Jupyter Notebook
    print("Monitoring on: ", symbols)
    print(result)  # Display the new result
    print("\n")
    
    return result

# Main loop that runs every 2 hours
while True:
    symbols = run_task()  # Get the symbols every 2 hours (list of up to 10 symbols)
    print("New symbols received. Monitoring begins...\n")
    print(f"Symbols to monitor ({len(symbols)} symbols): {symbols}")
    
    trade_taken = False  # Flag to track if any trade was taken
    
    # Try each symbol one by one
    for i, symbol in enumerate(symbols):
        print(f"\n--- Checking symbol {i+1}/{len(symbols)}: {symbol} ---")
        
        # Continuous 10-second updates with fetched data for current symbol
        symbol_checked = False
        
        while not symbol_checked:
            out = fetch_and_display_data_([symbol])  # Pass single symbol as list
            
            # Check if the DataFrame is not empty and get the last row
            if not out.empty:
                # First condition: Check if buy_signal sum is <= 15
                if out['buy_signal'].sum() <= 15:
                    print(f"Buy signal sum condition met for {symbol}: {out['buy_signal'].sum()}")
                    
                    # Second condition: Check price above ZLSMA and buy signal
                    if out['close'].iloc[-1] > out['zlsma_200'].iloc[-1] and out['buy_signal'].iloc[-1] == True:
                        print(f"✅ All conditions met! Taking trade for: {symbol}")
                        #bot = SimpleATRTradingBot()
                        #result = bot.buy_signal(symbol, 10)
                        #status = bot.get_position_status()
                        print(f"✅ Trade Taken: {symbol}")
                        print("Running trade for 2 hours...")
                        trade_taken = True
                        symbol_checked = True  # Move to next phase
                        break  # Exit the symbol checking loop
                    else:
                        print(f"Buy signal sum ok but no buy signal detected for {symbol}")
                        symbol_checked = True  # Try next symbol
                else:
                    print(f"Buy signal sum > 15 for {symbol} (sum: {out['buy_signal'].sum()}), trying next symbol...")
                    symbol_checked = True  # Try next symbol
            else:
                print(f"The DataFrame is empty for {symbol}. No data available.")
                symbol_checked = True  # Try next symbol
        
        # If trade was taken, break out of symbol iteration
        if trade_taken:
            break
    
    # If no trade was taken with any symbol, refresh for new symbols
    if not trade_taken:
        print("\n❌ No trades taken with any symbols. Refreshing for new symbols...")
        continue  # Skip the 2-hour sleep and get new symbols immediately
    
    # If trade was taken, monitor for 2 hours with 10-second intervals
    print(f"\n🔄 Monitoring trade for 2 hours...")
    monitoring_start = time.time()
    
    while time.time() - monitoring_start < 2 * 60 * 60:  # 2 hours
        # You can add monitoring logic here if needed
        time.sleep(10)  # Wait for 10 seconds before next check
    
    print("2-hour monitoring period completed. Getting new symbols...")

In [ ]:

def pick_best_coin():
    scalping_filter = BinanceAllUSDTScalpingFilter(
        max_workers=8,
        delay_between_requests=0.1,
        weight_profile='volatile'  # Options: 'balanced', 'volatile', 'trending'
    )

    print("Scanning ALL Binance USDT pairs for scalping opportunities...")

    # get a list of dicts (or records)
    best_coins = scalping_filter.filter_all_usdt_pairs(
        min_volume=50_000,
        top_n=30,
        use_parallel=True,
        volume_filter_first=False,
        use_percentile_volume=True
    )

    # turn into a DataFrame
    df = pd.DataFrame(best_coins)
    df = df[df['current_price'] <= 50]

    # base filters: uptrend + cheap coins
    #base_mask = (df['trend_direction'] == 1) & (df['current_price'] <= 50)

    # nothing matched
    return df["symbol"].tolist()

In [ ]:
bot = SimpleATRTradingBot()
result = bot.buy_signal('BTC', 1000)
status = bot.get_position_status()